# Evaluating a learned PDFA

This tutorial uses a small synthetic dataset to demonstrate how to:
1. Define separate training and test sequences;
2. Learn a PDFA using the training sequences;
3. Construct sequence probability tables;
4. Calculate evaluation metrics for the learned PDFA;
5. Evaluate the PDFA using a single function.

## Set up the tutorial

The first step is to import the pdfa_learning package. This can be done using the following code:

In [1]:
from pathlib import Path
import sys

repository_root = Path.cwd().parents[1]

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import pdfa_learning as pl

## 1. Define separate training and test sequences

To evaluate a learned PDFA, the observed sequences should be separated into training and test samples.

The training sequences are used to construct and learn the PDFA. The test sequences are then used to assess how closely the probabilities assigned by the learned PDFA correspond to sequences that were not used during the learning process.

We will use the same synthetic sample as the basic usage tutorial for the training sequences:

In [2]:
training_sequences = [ 
    "0", "0", "0", "0", "0", "0", "0", "0", 
    "01", "01", "01", "01", "01", "01", "01", 
    "10", "10", "10", "10", "10", 
    "11", "11", "11", "11", "11", 
    "12", "12", 
    "1", "1", "1", 
]

We can also define a separate sample of test sequences:

In [3]:
test_sequences = [
    "0", "0", "0", 
    "01", "01", 
    "10", "10", 
    "11", 
    "12", 
    "1", 
    "00", 
    "101",
]

The test sample contains both sequences that were observed during training and sequences that were not present in the training sample.

In particular, `"00"` and `"101"` were not included in training_sequences. These sequences allow us to examine whether the learned PDFA assigns probability to sequences outside the observed training sample.

All symbols present in the test sequences must belong to the alphabet learned from the training sequences.

## 2. Learn a PDFA using the training sequences

The alphabet should be derived from the training sequences:

In [4]:
alphabet = pl.get_alphabet(training_sequences)
alphabet

['0', '1', '2']

Then we can learn a PDFA using the training sequences and the derived alphabet:

In [10]:
states = pl.get_initial_states(training_sequences)

prefix_tree_acceptor = pl.get_transition_matrix(
    training_sequences,
    alphabet,
)

learned_dfa, learned_states, tracking = pl.alergia(
    prefix_tree_acceptor,
    states,
    alphabet,
    alpha=0.2,
    method="carrasco",
)

pdfa = pl.probability_transition_matrix(
    learned_dfa,
    learned_states,
    alphabet,
)

The resulting probability matrix represents the PDFA that will be evaluated against the held-out test sequences.

It is important that the test sequences were not included when constructing the prefix-tree acceptor or applying ALERGIA.

## 3. Construct sequence probability tables

The `get_sequence_probability_table` function compares the empirical probabilities of observed sequences with the probabilities assigned by the learned PDFA.

Before applying the function, it is useful to describe its parameters:
- `sequences`: The observed sequences for which probabilities will be calculated.
- `probability_matrix`: The probability transition matrix representing the learned PDFA.
- `alphabet`: The alphabet associated with the probability transition matrix.

The function returns one row for each unique observed sequence.

We will first construct a probability table for the testing sequences:

In [18]:
test_probability_table = pl.get_sequence_probability_table(
    sequences=test_sequences,
    probability_matrix=pdfa,
    alphabet=alphabet,
)

test_probability_table

,sequence,count,empirical_probability,pdfa_probability
0,0,3,0.250000,0.243243
1,01,2,0.166667,0.162162
2,10,2,0.166667,0.032871
3,11,1,0.083333,0.162162
4,12,1,0.083333,0.013148
5,1,1,0.083333,0.243243
6,00,1,0.083333,0.032871
7,101,1,0.083333,0.021914


The returned table contains the following columns:
- `sequence`: The unique observed sequence.
- `count`: The number of times that sequence occurs in the supplied sample.
- `empirical_probability`: The relative frequency of the sequence in the supplied sample.
- `pdfa_probability`: The probability assigned to the sequence by the learned PDFA.

The test table allows the empirical probabilities observed in the held-out data to be compared directly with the probabilities assigned by the learned PDFA.

A PDFA probability of zero indicates that the learned automaton cannot generate that sequence. A positive value indicates that the sequence is supported by the learned model, even when the sequence was not observed during training.

The empirical probabilities in the test table should sum to one:

In [ ]:
test_probability_table["empirical_probability"].sum()

1.0

The sum of the PDFA probabilities within the test table may be less than one:

In [ ]:
test_probability_table["pdfa_probability"].sum()

0.9116143170197223

We can construct the corresponding table for the training sequences as follows:

In [17]:
train_probability_table = pl.get_sequence_probability_table(
    sequences=training_sequences,
    probability_matrix=pdfa,
    alphabet=alphabet,
)

train_probability_table

,sequence,count,empirical_probability,pdfa_probability
0,0,8,0.266667,0.243243
1,01,7,0.233333,0.162162
2,10,5,0.166667,0.032871
3,11,5,0.166667,0.162162
4,12,2,0.066667,0.013148
5,1,3,0.100000,0.243243


The empirical probabilities in the training table sum to one because they represent the complete empirical distribution over the supplied training sample.

The PDFA probabilities do not necessarily sum to one within this table. The learned PDFA may assign some of its probability mass to sequences that were not observed during training.

### 4. Calculate evaluation metrics for the learned PDFA

The `calculate_pdfa_metrics` function calculates a collection of evaluation measures from the training and test probability tables.

Note that this is not an exhaustive list of possible evaluation measures. The function is designed to be easily extended with additional measures.

The test probability table is supplied first, followed by the training probability table:

In [19]:
metrics = pl.calculate_pdfa_metrics(
    test_probability_table=test_probability_table, 
    train_probability_table=train_probability_table, 
) 

metrics

{'mean_absolute_distance': 0.07073289505721939,
 'jensen_shannon_divergence': 0.08397246454204978,
 'sequence_coverage': 1.0,
 'probability_mass_coverage': 1.0,
 'training_sequence_pdfa_mass': 0.8568298027757486,
 'outside_training_sequence_pdfa_mass': 0.1431701972242514,
 'unseen_test_sequence_proportion': 0.25,
 'unseen_test_empirical_mass': 0.16666666666666666,
 'unseen_test_pdfa_mass': 0.05478451424397371}

The returned dictionary can be printed in a more readable format as follows:

In [20]:
for metric, value in metrics.items(): 
    print(f"{metric}: {value:.3f}")

mean_absolute_distance: 0.071
jensen_shannon_divergence: 0.084
sequence_coverage: 1.000
probability_mass_coverage: 1.000
training_sequence_pdfa_mass: 0.857
outside_training_sequence_pdfa_mass: 0.143
unseen_test_sequence_proportion: 0.250
unseen_test_empirical_mass: 0.167
unseen_test_pdfa_mass: 0.055


The evaluation measures can be divided into three broad groups:
- Distributional fit;
- Coverage of the test sequences;
- Allocation of probability mass.

### 4.1 Distributional fit

The following measures compare the empirical distribution in the test sample with the probabilities assigned by the learned PDFA:
- `mean_absolute_distance`: The mean absolute difference between the empirical test probabilities and the unnormalised probabilities assigned by the PDFA. Smaller values indicate closer agreement.
- `jensen_shannon_divergence`: The divergence between the empirical and PDFA distributions over the sequences observed in the test sample. The PDFA probabilities are normalised over the observed test support before this measure is calculated. A value of zero indicates identical distributions, while larger values indicate greater differences.

These measures assess different aspects of fit. The mean absolute distance retains information about the total probability mass assigned to the observed test sequences, while the Jensen-Shannon divergence compares the relative shapes of the two distributions after normalisation.

### 4.2 Coverage of the test sequences

The following measures describe how much of the held-out test sample is supported by the learned PDFA:
- `sequence_coverage`: The proportion of unique test sequences assigned a non-zero probability by the PDFA.
- `probability_mass_coverage`: The proportion of the empirical test probability mass belonging to sequences assigned a non-zero probability by the PDFA.

These values may differ because `sequence_coverage` gives equal weight to each unique sequence, whereas `probability_mass_coverage` accounts for how frequently each sequence occurs in the test sample.

For example, failing to cover one frequently observed sequence will have a larger effect on `probability_mass_coverage` than failing to cover one rare sequence.

### 4.3 Allocation of probability mass

The remaining measures describe where the learned PDFA allocates its probability mass:
- `training_sequence_pdfa_mass`: The total PDFA probability assigned to unique sequences observed during training.
- `outside_training_sequence_pdfa_mass`: The remaining PDFA probability mass assigned outside the unique training sequences.
- `unseen_test_sequence_proportion`: The proportion of unique test sequences that were not observed during training.
- `unseen_test_empirical_mass`: The empirical test probability mass belonging to sequences that were not observed during training.
- `unseen_test_pdfa_mass`: The total PDFA probability assigned to test sequences that were not observed during training.

The training and outside-training probability masses describe how strongly the learned model remains concentrated on the observed training sample.

The unseen test measures instead focus on the subset of held-out sequences that provide direct evidence of generalisation beyond the training data.

## 5. Evaluate the PDFA using a single function

The `evaluate_pdfa` function provides a convenient way to perform the same evaluation without first constructing the two probability tables manually.

It accepts the test sequences, training sequences, probability matrix, and alphabet:

In [21]:
evaluation = pl.evaluate_pdfa( 
    test_sequences=test_sequences, 
    train_sequences=training_sequences, 
    probability_matrix=pdfa, 
    alphabet=alphabet, 
) 

evaluation

{'mean_absolute_distance': 0.07073289505721939,
 'jensen_shannon_divergence': 0.08397246454204978,
 'sequence_coverage': 1.0,
 'probability_mass_coverage': 1.0,
 'training_sequence_pdfa_mass': 0.8568298027757486,
 'outside_training_sequence_pdfa_mass': 0.1431701972242514,
 'unseen_test_sequence_proportion': 0.25,
 'unseen_test_empirical_mass': 0.16666666666666666,
 'unseen_test_pdfa_mass': 0.05478451424397371}

The returned dictionary contains the same evaluation measures as the result produced by `calculate_pdfa_metrics`.

The two evaluation approaches are therefore suited to slightly different purposes:
- `evaluate_pdfa` is convenient when only the final evaluation measures are required.
- `get_sequence_probability_table` followed by `calculate_pdfa_metrics` is useful when the individual sequence probabilities also need to be inspected, reported, or analysed.

## Summary

In this tutorial, we learned a PDFA using a sample of training sequences and evaluated it against a separate held-out test sample.

We constructed sequence probability tables to compare empirical and PDFA probabilities, calculated measures of distributional fit and sequence coverage, and examined how the learned model allocated probability mass within and outside the training sample.

Finally, we used the `evaluate_pdfa` convenience function to perform the complete evaluation in a single step.

Candidate values of `alpha` may be compared using the same evaluation process and a separate validation sample. The final test sample should normally be retained for evaluating the selected model rather than repeatedly used during model selection.

When dealing with large datasets, you may find it useful to use the `train_test_split` function from the `sklearn.model_selection` module to create training and test samples from a single dataset. 

Please note though that `sklearn` is not a required dependency of the `pdfa_learning` package, so it is not imported or used within the package itself. You will need to install `sklearn` separately if you wish to use it for splitting your data.